In [ ]:
import os
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torch.optim as optim
from torch.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt

# ==========================================
# 1. IDENTICAL LOSS FUNCTIONS
# ==========================================
class SISDRLoss(nn.Module):
    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps
    def forward(self, preds, targets):
        if preds.dim() == 3: preds = preds.squeeze(1)
        if targets.dim() == 3: targets = targets.squeeze(1)
        preds   = preds - preds.mean(dim=-1, keepdim=True)
        targets = targets - targets.mean(dim=-1, keepdim=True)
        alpha         = (preds * targets).sum(-1, keepdim=True) / (targets.pow(2).sum(-1, keepdim=True) + self.eps)
        target_scaled = alpha * targets
        noise         = preds - target_scaled
        sisdr = 10 * torch.log10(
            (target_scaled.pow(2).sum(-1) + self.eps) /
            (noise.pow(2).sum(-1) + self.eps)
        )
        return -sisdr.mean()

class MRSTFTLoss(nn.Module):
    def __init__(self, fft_sizes=[512, 1024, 2048], hop_sizes=[128, 256, 512], win_lengths=[512, 1024, 2048]):
        super().__init__()
        self.fft_sizes   = fft_sizes
        self.hop_sizes   = hop_sizes
        self.win_lengths = win_lengths
    def forward(self, preds, targets):
        if preds.dim() == 3: preds = preds.squeeze(1)
        if targets.dim() == 3: targets = targets.squeeze(1)
        loss = 0.0
        for n_fft, hop, win in zip(self.fft_sizes, self.hop_sizes, self.win_lengths):
            window = torch.hann_window(win).to(preds.device)
            x_mag  = torch.abs(torch.stft(preds, n_fft=n_fft, hop_length=hop, win_length=win, window=window, return_complex=True, pad_mode='constant'))
            y_mag  = torch.abs(torch.stft(targets, n_fft=n_fft, hop_length=hop, win_length=win, window=window, return_complex=True, pad_mode='constant'))
            sc_loss  = torch.norm(y_mag - x_mag, p='fro') / (torch.norm(y_mag, p='fro') + 1e-7)
            mag_loss = F.l1_loss(torch.log(x_mag + 1e-7), torch.log(y_mag + 1e-7))
            loss += sc_loss + mag_loss
        return loss / len(self.fft_sizes)

class HybridLoss(nn.Module):
    def __init__(self, weight_stft=0.4, weight_sdr=0.6):
        super().__init__()
        self.sisdr_loss  = SISDRLoss()
        self.mrstft_loss = MRSTFTLoss()
        self.weight_stft = weight_stft
        self.weight_sdr  = weight_sdr
    def forward(self, preds, targets):
        loss_sdr  = self.sisdr_loss(preds, targets)
        loss_stft = self.mrstft_loss(preds, targets)
        total     = (self.weight_stft * loss_stft) + (self.weight_sdr * loss_sdr)
        return total, loss_sdr, loss_stft

# ==========================================
# 2. IDENTICAL DATALOADER (BATCH SIZE CONTROL)
# ==========================================
class FastOfflineDataset(Dataset):
    def __init__(self, split_dir, target_length=128000):
        self.mix_dir       = os.path.join(split_dir, 'mix')
        self.target_dir    = os.path.join(split_dir, 'target')
        self.target_length = target_length
        all_files          = sorted([f for f in os.listdir(self.mix_dir) if f.endswith('.wav')])
        self.filenames     = [f for f in all_files if os.path.exists(os.path.join(self.target_dir, f))]

    def __len__(self):
        return len(self.filenames)

    def _pad_or_crop(self, w):
        if w.shape[1] > self.target_length:
            return w[:, :self.target_length]
        elif w.shape[1] < self.target_length:
            return F.pad(w, (0, self.target_length - w.shape[1]))
        return w

    def __getitem__(self, idx):
        fname      = self.filenames[idx]
        mix, _     = torchaudio.load(os.path.join(self.mix_dir, fname))
        target, _  = torchaudio.load(os.path.join(self.target_dir, fname))
        if mix.shape[0] > 1: mix = mix.mean(0, keepdim=True)
        if target.shape[0] > 1: target = target.mean(0, keepdim=True)
        return self._pad_or_crop(mix), self._pad_or_crop(target)

def get_dataloaders(dataset_root, batch_size=4, num_workers=2): # BATCH SIZE 4 SET HERE
    train_ds = FastOfflineDataset(os.path.join(dataset_root, 'train'))
    val_ds   = FastOfflineDataset(os.path.join(dataset_root, 'val'))
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
    return train_loader, val_loader

In [ ]:
# ==========================================
# 3. YOUR PROPOSED ARCHITECTURE
# ==========================================
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1    = nn.Conv2d(in_channels,  out_channels, 3, padding=1)
        self.bn1      = nn.BatchNorm2d(out_channels)
        self.conv2    = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.bn2      = nn.BatchNorm2d(out_channels)
        self.se       = SEBlock(out_channels)
        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1),
                nn.BatchNorm2d(out_channels)
            )
    def forward(self, x):
        residual = self.shortcut(x)
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.se(out)
        out += residual
        return F.relu(out)

class ASPP(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1    = nn.Conv2d(in_channels, out_channels, 1)
        self.conv2    = nn.Conv2d(in_channels, out_channels, 3, padding=6,  dilation=6)
        self.conv3    = nn.Conv2d(in_channels, out_channels, 3, padding=12, dilation=12)
        self.conv4    = nn.Conv2d(in_channels, out_channels, 3, padding=18, dilation=18)
        self.out_conv = nn.Conv2d(out_channels * 4, out_channels, 1)
    def forward(self, x):
        x1 = F.relu(self.conv1(x))
        x2 = F.relu(self.conv2(x))
        x3 = F.relu(self.conv3(x))
        x4 = F.relu(self.conv4(x))
        return self.out_conv(torch.cat([x1, x2, x3, x4], dim=1))

class ResUNetPlusPlus(nn.Module):
    def __init__(self, n_fft=1024, hop_length=256):
        super().__init__()
        self.n_fft      = n_fft
        self.hop_length = hop_length
        self.window     = nn.Parameter(torch.hann_window(n_fft), requires_grad=False)

        # Encoder
        self.enc1  = ResidualBlock(2,   32)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2  = ResidualBlock(32,  64)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3  = ResidualBlock(64,  128)
        self.pool3 = nn.MaxPool2d(2)

        # ASPP Bottleneck
        self.aspp  = ASPP(128, 256)

        # Decoder
        self.up1   = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec1  = ResidualBlock(256, 128)
        self.up2   = nn.ConvTranspose2d(128,  64, 2, stride=2)
        self.dec2  = ResidualBlock(128,  64)
        self.up3   = nn.ConvTranspose2d(64,   32, 2, stride=2)
        self.dec3  = ResidualBlock(64,   32)

        self.out_mask = nn.Conv2d(32, 2, 1)

    def forward(self, waveform):
        if waveform.dim() == 3:
            waveform = waveform.squeeze(1)

        # STFT to Complex Spectrogram
        stft_out   = torch.stft(waveform, n_fft=self.n_fft, hop_length=self.hop_length,
                                window=self.window, return_complex=True, pad_mode='constant')
        real_part  = stft_out.real.unsqueeze(1)
        imag_part  = stft_out.imag.unsqueeze(1)
        x          = torch.cat([real_part, imag_part], dim=1)

        orig_F, orig_T = x.shape[2], x.shape[3]
        pad_F = (8 - (orig_F % 8)) % 8
        pad_T = (8 - (orig_T % 8)) % 8
        x = F.pad(x, (0, pad_T, 0, pad_F))

        # Forward Pass
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))

        bottleneck = self.aspp(self.pool3(e3))

        d1 = self.dec1(torch.cat([self.up1(bottleneck), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d1),         e2], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d2),         e1], dim=1))

        mask   = self.out_mask(d3)
        mask   = mask[:, :, :orig_F, :orig_T]
        x_orig = x[:, :, :orig_F, :orig_T]

        # Complex Multiplication
        x_real, x_imag = x_orig[:, 0], x_orig[:, 1]
        m_real, m_imag = mask[:, 0],   mask[:, 1]

        out_real    = (x_real * m_real) - (x_imag * m_imag)
        out_imag    = (x_real * m_imag) + (x_imag * m_real)
        out_complex = torch.complex(out_real, out_imag)

        # ISTFT Reconstruction
        return torch.istft(out_complex, n_fft=self.n_fft, hop_length=self.hop_length,
                           window=self.window, length=waveform.shape[1]).unsqueeze(1)

In [ ]:
# ==========================================
# 4. TRAINING LOOP & PLOT GENERATION
# ==========================================
def train_resunet_plus_plus():
    # 🚀 A100 SPEED UP: Enable CUDNN Benchmark and TF32
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"🚀 Launching Proposed ResUNet++ on: {device}")

    # --- SAVE DIRECTORIES ---
    BASE_DIR       = '/content/drive/MyDrive'
    CHECKPOINT_DIR = os.path.join(BASE_DIR, 'ResUNetPlusPlus_Results')
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)

    # ⚠️ THIS IS THE EXACT PATH THAT WORKED FOR YOU EARLIER
    DATASET_ROOT = "/content/content/synthetic_dataset_v2"

    # 🚀 A100 SPEED UP: Increased batch_size to 16 and num_workers to 8
    train_loader, val_loader = get_dataloaders(DATASET_ROOT, batch_size=16, num_workers=8)

    model     = ResUNetPlusPlus().to(device)
    criterion = HybridLoss(weight_stft=0.4, weight_sdr=0.6).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
    epochs    = 50
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    scaler    = GradScaler('cuda')

    best_val_sdr    = float('-inf')
    checkpoint_path = os.path.join(CHECKPOINT_DIR, 'best_resunet_plus_plus.pth')
    history_path    = os.path.join(CHECKPOINT_DIR, 'resunet_history.json')
    history         = {'train_loss': [], 'val_loss': [], 'train_sdr': [], 'val_sdr': []}

    # Early stopping parameters
    patience = 10
    no_improve_epochs = 0

    for epoch in range(1, epochs + 1):
        # --- Train ---
        model.train()
        train_loss, train_sdr = 0.0, 0.0
        loop = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs} [Train]", leave=False)
        for mixed, target in loop:
            mixed, target = mixed.to(device), target.to(device)
            optimizer.zero_grad()
            with autocast('cuda'):
                preds = model(mixed)
                loss, loss_sdr, _ = criterion(preds, target)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item()
            train_sdr  += -loss_sdr.item()
            loop.set_postfix(loss=f"{loss.item():.4f}", sdr=f"{-loss_sdr.item():.3f}")

        avg_train_loss = train_loss / len(train_loader)
        avg_train_sdr  = train_sdr  / len(train_loader)

        # --- Validate ---
        model.eval()
        val_loss, val_sdr = 0.0, 0.0
        with torch.no_grad():
            for mixed, target in tqdm(val_loader, desc=f"Epoch {epoch}/{epochs} [Val]", leave=False):
                mixed, target = mixed.to(device), target.to(device)
                with autocast('cuda'):
                    preds = model(mixed)
                    loss, loss_sdr, _ = criterion(preds, target)
                val_loss += loss.item()
                val_sdr  += -loss_sdr.item()

        avg_val_loss = val_loss / len(val_loader)
        avg_val_sdr  = val_sdr  / len(val_loader)

        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['train_sdr'].append(avg_train_sdr)
        history['val_sdr'].append(avg_val_sdr)

        # Save history JSON continuously
        with open(history_path, 'w') as f: json.dump(history, f)
        scheduler.step()

        print(f"Epoch [{epoch}/{epochs}] | Train SDR: {avg_train_sdr:.3f} dB | Val SDR: {avg_val_sdr:.3f} dB")

        # Save Best Model
        if avg_val_sdr > best_val_sdr:
            best_val_sdr = avg_val_sdr
            torch.save(model.state_dict(), checkpoint_path)
            print(f"   🌟 Best ResUNet++ saved | Val SI-SDR: {best_val_sdr:.3f} dB")
            no_improve_epochs = 0
        else:
            no_improve_epochs += 1
            print(f"   ⚠️ No improvement for {no_improve_epochs} epoch(s).")
            if no_improve_epochs >= patience:
                print(f"🛑 Early stopping triggered! No improvement for {patience} epochs.")
                break

    # ==========================================
    # 🎉 TRAINING COMPLETE - GENERATE PLOTS
    # ==========================================
    print("Training Complete! Generating plots...")

    # 1. Loss Plot
    plt.figure(figsize=(10, 5))
    plt.plot(history['train_loss'], label='Train Loss', color='blue')
    plt.plot(history['val_loss'], label='Val Loss', color='orange')
    plt.title('Proposed ResUNet++: Hybrid Loss Over Epochs')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(CHECKPOINT_DIR, 'loss_plot.png'))
    plt.close()

    # 2. SI-SDR Plot
    plt.figure(figsize=(10, 5))
    plt.plot(history['train_sdr'], label='Train SI-SDR', color='green')
    plt.plot(history['val_sdr'], label='Val SI-SDR', color='red')
    plt.title('Proposed ResUNet++: SI-SDR Improvement Over Epochs')
    plt.xlabel('Epochs')
    plt.ylabel('SI-SDR (dB)')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(CHECKPOINT_DIR, 'sdr_plot.png'))
    plt.close()

    print(f"✅ All files (Model, JSON History, and Plots) securely saved to {CHECKPOINT_DIR}")

# 🚀 RUN IT
train_resunet_plus_plus()